# Scipy L-BFGS-B Tutorial

This tutorial demonstrates how to minimize an objective with `ls_bayesian`'s `optimization` subpackage, using its scipy-backed algorithm,
`ScipyLBFGSBOptimizer` (`ls_bayesian.optimization.algorithms.scipy_lbfgs_b`), which wraps
`scipy.optimize.minimize(method="L-BFGS-B")`. This is the right choice for problems in
the standard Euclidean geometry, e.g. MAP estimation with a `LogPosterior` whose
gradient is already expressed as a Euclidean gradient. For problems that require a
custom inner product -- e.g. a Cameron-Martin metric induced by a Bayesian prior's
covariance -- see the companion `custom_lbfgs` tutorial, which uses a metric-generic
backend instead.

## The `OptimizationModel` interface

Every backend in `optimization` minimizes an
[`OptimizationModel`][ls_bayesian.optimization.model.OptimizationModel]: an objective
bundled with the inner-product space its gradient is expressed in
(`evaluate_cost`, `evaluate_gradient`, `evaluate_hessian_vector_product`,
`evaluate_inner_product`). `ScipyLBFGSBOptimizer` never calls `evaluate_hessian_vector_product`
or `evaluate_inner_product` -- L-BFGS-B only needs gradients, and scipy's implementation
is hardcoded to the Euclidean inner product -- but the model must still implement the
full interface, since `OptimizationModel` is shared by every backend in the subpackage.

In [ ]:
from typing import override

import matplotlib.pyplot as plt
import numpy as np

from ls_bayesian.common.logging import BaseLogger, LoggerSettings
from ls_bayesian.optimization.algorithms.scipy_lbfgs_b import (
    ScipyLBFGSBOptimizer,
    ScipyLBFGSBSettings,
)
from ls_bayesian.optimization.model import OptimizationModel

rng = np.random.default_rng(0)
PARAMETER_DIM = 6

## Objective: the Rosenbrock function

As a concrete, non-convex test objective we use the standard $n$-dimensional Rosenbrock
function

$$
\begin{equation*}
    f(x) = \sum_{i=1}^{n-1} \left[100 (x_{i+1} - x_i^2)^2 + (1 - x_i)^2\right],
\end{equation*}
$$

whose unique minimizer is $x^* = (1, \dots, 1)$. Its narrow, curved valley makes it a
standard stress test for quasi-Newton methods such as L-BFGS-B.

In [ ]:
class RosenbrockModel(OptimizationModel):
    """Standard n-dimensional Rosenbrock function, minimizer at all-ones, standard
    Euclidean inner product. No Hessian-vector product support."""

    @override
    def evaluate_cost(self, parameter_vector: np.ndarray) -> float:
        x = parameter_vector[:-1]
        y = parameter_vector[1:]
        return float(np.sum(100.0 * (y - x**2) ** 2 + (1.0 - x) ** 2))

    @override
    def evaluate_gradient(self, parameter_vector: np.ndarray) -> np.ndarray:
        gradient = np.zeros_like(parameter_vector)
        x = parameter_vector[:-1]
        y = parameter_vector[1:]
        gradient[:-1] += -400.0 * x * (y - x**2) - 2.0 * (1.0 - x)
        gradient[1:] += 200.0 * (y - x**2)
        return gradient

    @override
    def evaluate_hessian_vector_product(
        self, parameter_vector: np.ndarray, direction_vector: np.ndarray
    ) -> np.ndarray:
        raise NotImplementedError

    @override
    def evaluate_inner_product(self, first_vector: np.ndarray, second_vector: np.ndarray) -> float:
        return float(np.dot(first_vector, second_vector))


model = RosenbrockModel()
minimizer = np.ones(PARAMETER_DIM)

## Configuring and running the optimizer

[`ScipyLBFGSBSettings`][ls_bayesian.optimization.algorithms.scipy_lbfgs_b.ScipyLBFGSBSettings]
collects the options passed through to `scipy.optimize.minimize`; its defaults reproduce
scipy's own defaults for `method="L-BFGS-B"`. Here we tighten the gradient tolerance a
little to see a cleaner convergence plot below. `ScipyLBFGSBOptimizer.run` (inherited from
[`BaseOptimizer`][ls_bayesian.optimization.optimizer.BaseOptimizer]) takes the initial
guess and the model, and optionally reports iteration-by-iteration progress to a
[`BaseLogger`][ls_bayesian.common.logging.BaseLogger].

In [ ]:
settings = ScipyLBFGSBSettings(relative_gradient_tolerance=1e-8)
logger = BaseLogger(LoggerSettings(), prefix="lbfgs")
optimizer = ScipyLBFGSBOptimizer(settings, logger=logger)

initial_guess = np.full(PARAMETER_DIM, -1.2)
initial_guess[1::2] = 1.0

result = optimizer.run(initial_guess, model)

## Inspecting the result

`run` returns an
[`OptimizationResult`][ls_bayesian.optimization.optimizer.OptimizationResult]: the
minimizer found, whether the backend reports convergence, a human-readable status
message, the number of iterations, and the loss/gradient-norm history recorded at every
accepted iterate.

In [ ]:
print(f"success: {result.success}")
print(f"status: {result.status_message}")
print(f"iterations: {result.num_iterations}")
print(f"distance to minimizer: {np.linalg.norm(result.result - minimizer):.3e}")

## Convergence plot

`loss_history` and `gradient_norm_history` record one value per accepted iteration (not
per internal line-search trial), letting us plot the optimizer's progress directly.

In [ ]:
fig, ax = plt.subplots()
ax.semilogy(result.loss_history, label="loss")
ax.semilogy(result.gradient_norm_history, label="gradient norm")
ax.set_xlabel("iteration")
ax.set_ylabel("value (log scale)")
ax.set_title("L-BFGS-B convergence on the Rosenbrock function")
ax.legend()
plt.show()

## Verifying convergence to the known minimizer

The Rosenbrock function's minimizer is known analytically, so we can check the result
directly rather than only trusting scipy's own convergence flag.

In [ ]:
np.testing.assert_allclose(result.result, minimizer, atol=1e-4)
print("Converged to the known minimizer within tolerance.")